# Optimizers

> bioMONAI optimizers

This section exposes the core optimization components integrated within bioMONAI, wrapping standard optimization structures (such as `OptimWrapper` and `Adam`) to streamline model training workflows.

In [ ]:
#| default_exp optimizers

In [ ]:
#| hide
from nbdev.showdoc import *
from fastcore.test import *

from torch import randn as torchrandn

In [ ]:
#| export

# =================================
# Scientific / data
# =================================
# import numpy as np
# import pandas as pd

# =================================
# PyTorch
# =================================
import torch.optim as toptim

# =================================
# fastai
# =================================
from fastai.optimizer import OptimWrapper

# =================================
# bioMONAI
# =================================
from bioMONAI.backend import get_backend
from bioMONAI.utils import *

## Base Functions
### Backends

In [ ]:
#| export   

OPTIMIZER_BACKENDS = {}


def register_optimizer_backend(name):
    """Register an optimizer backend."""
    def decorator(cls):
        OPTIMIZER_BACKENDS[name] = cls
        return cls
    return decorator



### `OptimWrapper` Class Attributes & Parameter Reference

*A wrapper class for existing PyTorch optimizers to integrate them seamlessly into fastai workflows.*

| Parameter / Attribute | Type | Default | Description |
| :--- | :--- | :--- | :--- |
| **`params`** | `Tensor \| Iterable` | `None` | Model parameters. Don't set if using a built optimizer. |
| **`opt`** | `Callable \| torch.optim.Optimizer` | `None` | A torch optimizer constructor, or an already built optimizer. |
| **`hp_map`** | `dict` | `None` | A dictionary converting PyTorch optimizer keys to fastai's `Optimizer` keys. Defaults to `pytorch_hp_map`. |
| **`convert_groups`** | `bool` | `True` | Convert parameter groups from splitter or pass unaltered to `opt`. |
| **`**kwargs`** | `VAR_KEYWORD` | – | Additional keyword arguments passed to the optimizer. |

In [ ]:
#| export
OptimWrapper = OptimWrapper

In [ ]:
show_doc(OptimWrapper)

---

[source](https://github.com/fastai/fastai/blob/main/fastai/optimizer.py#LNone){target="_blank" style="float:right; font-size:smaller"}

### OptimWrapper

```python

def OptimWrapper(
    params:Tensor | Iterable=None, # Model parameters. Don't set if using a built optimizer
    opt:Callable | torch.optim.Optimizer=None, # A torch optimizer constructor, or an already built optimizer
    hp_map:dict=None, # A dictionary converting PyTorch optimizer keys to fastai's `Optimizer` keys. Defaults to `pytorch_hp_map`
    convert_groups:bool=True, # Convert parameter groups from splitter or pass unaltered to `opt`
    kwargs:VAR_KEYWORD
):


```

*A wrapper class for existing PyTorch optimizers*

### Backend abstractions

In [ ]:
#| export

class OptimizerBackend:
    """Base class for backend-specific optimizer adapters."""

    @classmethod
    def create(cls, optimizer_cls, params, *args, **kwargs):
        """
        Create a backend-specific optimizer.

        Parameters
        ----------
        optimizer_cls : type
            PyTorch optimizer class.
        params : iterable
            Parameters to optimize.
        *args
            Positional arguments passed to the optimizer.
        **kwargs
            Keyword arguments passed to the optimizer.

        Returns
        -------
        object
            Backend-specific optimizer.
        """
        raise NotImplementedError

In [ ]:
#| export

@register_optimizer_backend("torch")
class TorchOptimizerBackend(OptimizerBackend):
    """
    Backend adapter for PyTorch optimizers.

    This is the native optimizer backend. It instantiates the supplied
    PyTorch optimizer class directly and returns the resulting optimizer
    instance.

    Parameters
    ----------
    optimizer_cls : type
        PyTorch optimizer class to instantiate.
    params : iterable
        Parameters or parameter groups to optimize.
    *args
        Positional arguments passed to the optimizer.
    **kwargs
        Keyword arguments passed to the optimizer.

    Returns
    -------
    torch.optim.Optimizer
        Instantiated PyTorch optimizer.
    """

    @classmethod
    def create(cls, optimizer_cls, params, *args, **kwargs):
        """Instantiate and return the PyTorch optimizer."""
        return optimizer_cls(params, *args, **kwargs)


@register_optimizer_backend("fastai")
class FastaiOptimizerBackend(OptimizerBackend):
    """
    Backend adapter for fastai optimizers.

    The optimizer is first instantiated using its native PyTorch
    implementation and then wrapped with fastai's ``OptimWrapper``.
    This allows fastai training components to use the same PyTorch
    optimizer while exposing the interface expected by fastai.

    Parameters
    ----------
    optimizer_cls : type
        PyTorch optimizer class to instantiate.
    params : iterable
        Parameters or parameter groups to optimize.
    *args
        Positional arguments passed to the PyTorch optimizer.
    **kwargs
        Keyword arguments passed to the optimizer.

    Returns
    -------
    fastai.optimizer.OptimWrapper
        Fastai wrapper around the PyTorch optimizer.
    """

    @classmethod
    def create(cls, optimizer_cls, params, *args, **kwargs):
        """Instantiate a PyTorch optimizer and wrap it for fastai."""
        optimizer = optimizer_cls(params, *args, **kwargs)
        return OptimWrapper(optimizer)


@register_optimizer_backend("monai")
class MonaiOptimizerBackend(OptimizerBackend):
    """
    Backend adapter for MONAI optimizers.

    MONAI training workflows use PyTorch optimizers, so this adapter
    currently instantiates and returns the supplied PyTorch optimizer
    directly.

    Parameters
    ----------
    optimizer_cls : type
        PyTorch optimizer class to instantiate.
    params : iterable
        Parameters or parameter groups to optimize.
    *args
        Positional arguments passed to the optimizer.
    **kwargs
        Keyword arguments passed to the optimizer.

    Returns
    -------
    torch.optim.Optimizer
        Instantiated PyTorch optimizer.
    """

    @classmethod
    def create(cls, optimizer_cls, params, *args, **kwargs):
        """Instantiate and return the PyTorch optimizer for MONAI."""
        return optimizer_cls(params, *args, **kwargs)


@register_optimizer_backend("ignite")
class IgniteOptimizerBackend(OptimizerBackend):
    """
    Backend adapter for Ignite optimizers.

    Ignite training engines can use standard PyTorch optimizers.
    This adapter therefore currently returns the supplied PyTorch
    optimizer without additional wrapping.

    Parameters
    ----------
    optimizer_cls : type
        PyTorch optimizer class to instantiate.
    params : iterable
        Parameters or parameter groups to optimize.
    *args
        Positional arguments passed to the optimizer.
    **kwargs
        Keyword arguments passed to the optimizer.

    Returns
    -------
    torch.optim.Optimizer
        Instantiated PyTorch optimizer.
    """

    @classmethod
    def create(cls, optimizer_cls, params, *args, **kwargs):
        """Instantiate and return the PyTorch optimizer for Ignite."""
        return optimizer_cls(params, *args, **kwargs)


@register_optimizer_backend("keras")
class KerasOptimizerBackend(OptimizerBackend):
    """
    Backend adapter for Keras optimizers.

    This adapter currently uses the PyTorch optimizer implementation,
    matching the temporary backend strategy used by bioMONAI. It can
    later be replaced with native Keras optimizer construction without
    changing the public ``BioOptimizer`` interface.

    Parameters
    ----------
    optimizer_cls : type
        PyTorch optimizer class to instantiate.
    params : iterable
        Parameters or parameter groups to optimize.
    *args
        Positional arguments passed to the optimizer.
    **kwargs
        Keyword arguments passed to the optimizer.

    Returns
    -------
    torch.optim.Optimizer
        Instantiated PyTorch optimizer.
    """

    @classmethod
    def create(cls, optimizer_cls, params, *args, **kwargs):
        """Instantiate and return the PyTorch optimizer for Keras."""
        return optimizer_cls(params, *args, **kwargs)

### Base function

In [ ]:
#| export

class BioOptimizer:
    """
    Backend-independent interface for optimizers in bioMONAI.

    ``BioOptimizer`` provides a common interface for optimizers across
    supported bioMONAI backends. Each optimizer subclass defines a default
    implementation through ``_default`` and may optionally provide
    backend-specific implementations through attributes such as ``_torch``,
    ``_fastai``, ``_keras``, or ``_monai``.

    The active backend is selected globally with
    :func:`bioMONAI.set_backend` and is obtained through
    :func:`bioMONAI.get_backend`. Optimizers therefore do not expose a
    ``backend`` argument.

    Parameters
    ----------
    params : iterable
        Iterable of parameters to optimize.
    *args
        Positional arguments passed to the optimizer implementation.
    **kwargs
        Keyword arguments passed to the optimizer implementation.

    Attributes
    ----------
    params : iterable
        Parameters passed to the optimizer.
    optimizer : object
        Backend-specific optimizer instance.

    Class Attributes
    ----------------
    _default : callable
        Default optimizer implementation.
    _<backend> : callable, optional
        Backend-specific optimizer implementation. When defined, it takes
        precedence over ``_default`` for that backend.

    Examples
    --------
    Select the backend globally and instantiate an optimizer normally:

    >>> bioMONAI.set_backend("torch")
    >>> optimizer = Adam(model.parameters(), lr=1e-3)

    The same interface can be used with another backend:

    >>> bioMONAI.set_backend("fastai")
    >>> optimizer = Adam(model.parameters(), lr=1e-3)
    """

    _default = None

    def __init__(
        self,
        params,
        *args,
        **kwargs,
    ):
        self.params = params

        backend = get_backend()
        optimizer_cls = self._get_optimizer_cls(backend)

        self.optimizer = self._create_optimizer(
            backend,
            optimizer_cls,
            params,
            *args,
            **kwargs,
        )

    @classmethod
    def _get_optimizer_cls(cls, backend):
        """
        Return the optimizer implementation for ``backend``.

        A backend-specific implementation takes precedence over
        ``_default``.
        """
        optimizer_cls = getattr(cls, f"_{backend}", None)

        if optimizer_cls is None:
            optimizer_cls = cls._default

        if optimizer_cls is None:
            raise ValueError(
                f"No optimizer implementation for backend '{backend}' "
                f"and no default implementation is defined for "
                f"{cls.__name__}."
            )

        return optimizer_cls

    @classmethod
    def _create_optimizer(
        cls,
        backend,
        optimizer_cls,
        params,
        *args,
        **kwargs,
    ):
        """
        Create the optimizer using the selected backend adapter.
        """
        try:
            backend_cls = OPTIMIZER_BACKENDS[backend]
        except KeyError:
            raise ValueError(
                f"Unknown optimizer backend '{backend}'. "
                f"Available optimizer backends: "
                f"{list(OPTIMIZER_BACKENDS)}"
            ) from None

        return backend_cls.create(
            optimizer_cls,
            params,
            *args,
            **kwargs,
        )

## Optimizers

In [ ]:
#| export
class Adadelta(BioOptimizer):
    """
    Adadelta optimizer with adaptive per-parameter learning rates.

    Adadelta adapts the learning rate using a decaying average of past
    squared gradients and a decaying average of past squared parameter
    updates. Unlike Adagrad, it does not accumulate squared gradients
    indefinitely, which prevents the effective learning rate from
    continually shrinking.

    For gradient ``g_t``, the running averages are

    ``E[g²]_t = rho * E[g²]_{t-1} + (1 - rho) * g_t²``

    and

    ``E[Δθ²]_t = rho * E[Δθ²]_{t-1} + (1 - rho) * Δθ_t²``.

    The parameter update is approximately

    ``Δθ_t = -sqrt(E[Δθ²]_{t-1} + eps) /
             sqrt(E[g²]_t + eps) * g_t``

    followed by ``θ_t = θ_{t-1} + Δθ_t``.

    Parameters
    ----------
    lr : float, default=1.0
        Coefficient applied to the adaptive update. Although Adadelta
        is relatively insensitive to the choice of learning rate, it
        still scales the resulting parameter update.
    rho : float, default=0.9
        Decay factor used for the running averages of squared gradients
        and squared parameter updates. Larger values give longer memory.
    eps : float, default=1e-6
        Small constant added for numerical stability.
    weight_decay : float, default=0
        L2 penalty applied to the parameters.
    foreach : bool or None, default=None
        Whether to use the multi-tensor implementation when available.
    maximize : bool, default=False
        If ``True``, performs gradient ascent instead of gradient descent.
    capturable : bool, default=False
        Enables CUDA graph / ``torch.compile`` compatible state handling
        where supported.
    differentiable : bool, default=False
        If ``True``, records the optimizer operation as part of the
        autograd graph.
    """
    _default = toptim.Adadelta


class Adafactor(BioOptimizer):
    """
    Memory-efficient adaptive optimizer based on factorized second moments.

    Adafactor is designed primarily for large models where storing a full
    second-moment tensor for every parameter would consume substantial
    memory. For sufficiently large matrix-like parameters, the second
    moment is approximated using row and column statistics rather than
    storing one value for every element.

    The adaptive update is based on a normalized second-moment estimate
    followed by optional parameter-scale-dependent learning-rate scaling.
    In simplified form,

    ``θ_t = θ_{t-1} - η_t * g_t / (sqrt(V_t) + eps)``

    where ``V_t`` is the estimated second moment and ``η_t`` is the
    effective learning rate determined by the optimizer configuration.

    Parameters
    ----------
    lr : float or None, default=None
        Learning rate. When ``None``, Adafactor uses its relative-step
        learning-rate schedule.
    beta2_decay : float, default=-0.8
        Exponent controlling the time-dependent decay used to estimate
        the second moment.
    eps : tuple, default=(1e-30, 1e-3)
        Numerical-stability constants used by the second-moment
        computation and update scaling.
    d : float, default=1.0
        Constant controlling the scale of the relative learning-rate
        schedule.
    weight_decay : float, default=0.0
        Weight-decay coefficient.
    foreach : bool or None, default=None
        Whether to use the multi-tensor implementation when available.
    maximize : bool, default=False
        If ``True``, performs gradient ascent.
    capturable : bool, default=False
        Enables graph-capturable state handling where supported.
    """
    _default = toptim.Adafactor


class Adagrad(BioOptimizer):
    """
    Adaptive Gradient optimizer with a separate effective learning rate
    for each parameter.

    Adagrad accumulates the squared gradients seen during optimization
    and divides each parameter's gradient by the square root of this
    accumulated quantity:

    ``G_t = G_{t-1} + g_t²``

    ``θ_t = θ_{t-1} - lr * g_t / (sqrt(G_t) + eps)``.

    Consequently, frequently updated parameters receive progressively
    smaller effective learning rates, while rarely updated parameters
    retain relatively larger learning rates.

    Parameters
    ----------
    lr : float, default=1e-2
        Initial learning rate.
    lr_decay : float, default=0
        Learning-rate decay applied as a function of the optimization
        step count.
    weight_decay : float, default=0
        L2 penalty applied to the parameters.
    initial_accumulator_value : float, default=0
        Initial value of the accumulated squared-gradient state.
    eps : float, default=1e-10
        Numerical-stability constant added to the denominator.
    foreach : bool or None, default=None
        Whether to use the multi-tensor implementation when available.
    maximize : bool, default=False
        If ``True``, performs gradient ascent.
    differentiable : bool, default=False
        If ``True``, records optimizer operations in the autograd graph.
    """
    _default = toptim.Adagrad


class Adam(BioOptimizer):
    """
    Adaptive Moment Estimation optimizer.

    Adam maintains exponentially decaying estimates of both the first
    and second moments of the gradient:

    ``m_t = beta1 * m_{t-1} + (1 - beta1) * g_t``

    ``v_t = beta2 * v_{t-1} + (1 - beta2) * g_t²``.

    Bias-corrected estimates are used to compute the update:

    ``m̂_t = m_t / (1 - beta1^t)``

    ``v̂_t = v_t / (1 - beta2^t)``

    ``θ_t = θ_{t-1} - lr * m̂_t / (sqrt(v̂_t) + eps)``.

    Parameters
    ----------
    lr : float, default=1e-3
        Learning rate.
    betas : tuple of float, default=(0.9, 0.999)
        Decay rates for the first and second moment estimates.
    eps : float, default=1e-8
        Numerical-stability constant added to the denominator.
    weight_decay : float, default=0
        Weight-decay coefficient. For decoupled weight decay, use
        ``AdamW``.
    amsgrad : bool, default=False
        If ``True``, maintains the maximum historical second-moment
        estimate as in AMSGrad.
    foreach : bool or None, default=None
        Whether to use the multi-tensor implementation when available.
    maximize : bool, default=False
        If ``True``, performs gradient ascent.
    capturable : bool, default=False
        Enables graph-capturable optimizer state where supported.
    differentiable : bool, default=False
        If ``True``, records optimizer operations in the autograd graph.
    fused : bool or None, default=None
        Whether to use the fused implementation when supported by the
        selected device and dtype.
    """
    _default = toptim.Adam


class AdamW(BioOptimizer):
    """
    Adam optimizer with decoupled weight decay.

    AdamW separates weight decay from the adaptive gradient update.
    The Adam update is computed from the gradients, while the parameters
    are independently shrunk according to the weight-decay coefficient.

    Ignoring implementation details, the update can be written as

    ``θ_t = θ_{t-1} - lr * AdamUpdate(g_t)
           - lr * weight_decay * θ_{t-1}``.

    This decoupling makes the weight-decay parameter behave independently
    of the gradient normalization used by Adam.

    Parameters
    ----------
    lr : float, default=1e-3
        Learning rate.
    betas : tuple of float, default=(0.9, 0.999)
        Decay rates for the first- and second-moment estimates.
    eps : float, default=1e-8
        Numerical-stability constant.
    weight_decay : float, default=1e-2
        Coefficient controlling decoupled parameter shrinkage.
    amsgrad : bool, default=False
        If ``True``, uses the AMSGrad variant.
    foreach : bool or None, default=None
        Whether to use the multi-tensor implementation when available.
    maximize : bool, default=False
        If ``True``, performs gradient ascent.
    capturable : bool, default=False
        Enables graph-capturable state handling where supported.
    differentiable : bool, default=False
        If ``True``, records optimizer operations in the autograd graph.
    fused : bool or None, default=None
        Whether to use the fused implementation when supported.
    """
    _default = toptim.AdamW


class Adamax(BioOptimizer):
    """
    Adam variant using the infinity norm of past gradients.

    Adamax replaces Adam's exponentially weighted second moment with an
    exponentially weighted infinity norm. The state is updated as

    ``u_t = max(beta2 * u_{t-1}, |g_t|)``.

    The first-moment estimate is computed as in Adam and the parameter
    update uses the infinity-norm state:

    ``θ_t = θ_{t-1} - lr * m̂_t / (u_t + eps)``.

    Adamax can be numerically useful when gradients contain large or
    highly variable values.

    Parameters
    ----------
    lr : float, default=2e-3
        Learning rate.
    betas : tuple of float, default=(0.9, 0.999)
        Decay rates for the first moment and infinity-norm estimates.
    eps : float, default=1e-8
        Numerical-stability constant.
    weight_decay : float, default=0
        L2 weight-decay coefficient.
    foreach : bool or None, default=None
        Whether to use the multi-tensor implementation when available.
    maximize : bool, default=False
        If ``True``, performs gradient ascent.
    differentiable : bool, default=False
        If ``True``, records optimizer operations in the autograd graph.
    """
    _default = toptim.Adamax


class ASGD(BioOptimizer):
    """
    Averaged Stochastic Gradient Descent optimizer.

    ASGD performs stochastic gradient updates while maintaining a
    time-weighted average of the parameters after a configurable
    starting point. The averaged parameters can provide improved
    convergence behavior for some optimization problems.

    Parameters
    ----------
    lr : float, default=1e-2
        Initial learning rate.
    lambd : float, default=1e-4
        Regularization/decay coefficient controlling the learning-rate
        schedule.
    alpha : float, default=0.75
        Power used in the learning-rate schedule.
    t0 : float, default=1e6
        Number of iterations before averaging begins.
    weight_decay : float, default=0
        L2 weight-decay coefficient.
    foreach : bool or None, default=None
        Whether to use the multi-tensor implementation when available.
    maximize : bool, default=False
        If ``True``, performs gradient ascent.
    differentiable : bool, default=False
        If ``True``, records optimizer operations in the autograd graph.
    """
    _default = toptim.ASGD


class LBFGS(BioOptimizer):
    """
    Limited-memory BFGS quasi-Newton optimizer.

    LBFGS approximates second-order optimization without explicitly
    constructing the Hessian. It stores a limited history of parameter
    and gradient differences to approximate inverse-Hessian information.

    Unlike most PyTorch optimizers, LBFGS requires a ``closure`` passed
    to ``step``. The closure must recompute the model loss and gradients,
    because LBFGS may evaluate it multiple times during a single
    optimization step.

    LBFGS is generally better suited to relatively small models or
    optimization problems where accurate deterministic optimization is
    more important than minimizing per-step computation and memory.

    Parameters
    ----------
    lr : float, default=1
        Step-size multiplier.
    max_iter : int, default=20
        Maximum number of internal optimization iterations per
        ``step`` call.
    max_eval : int or None, default=None
        Maximum number of closure evaluations per step. If ``None``,
        PyTorch derives a value from ``max_iter``.
    tolerance_grad : float, default=1e-7
        Terminate when the maximum gradient magnitude falls below this
        threshold.
    tolerance_change : float, default=1e-9
        Terminate when the loss or parameter change becomes sufficiently
        small.
    history_size : int, default=100
        Number of previous updates retained for the inverse-Hessian
        approximation.
    line_search_fn : str or None, default=None
        Optional line-search algorithm. PyTorch currently supports
        ``"strong_wolfe"``.
    """
    _default = toptim.LBFGS


class NAdam(BioOptimizer):
    """
    Adam optimizer with Nesterov-style momentum.

    NAdam combines Adam's adaptive first- and second-moment estimates
    with a Nesterov-style momentum formulation. The resulting update
    uses information from the current gradient together with the
    momentum estimate to provide a look-ahead effect.

    Parameters
    ----------
    lr : float, default=2e-3
        Learning rate.
    betas : tuple of float, default=(0.9, 0.999)
        Decay rates for the first- and second-moment estimates.
    eps : float, default=1e-8
        Numerical-stability constant.
    weight_decay : float, default=0
        Weight-decay coefficient.
    momentum_decay : float, default=4e-3
        Decay applied to the Nesterov momentum schedule.
    decoupled_weight_decay : bool, default=False
        If ``True``, applies weight decay independently of the gradient
        update, giving AdamW-like behavior.
    foreach : bool or None, default=None
        Whether to use the multi-tensor implementation when available.
    maximize : bool, default=False
        If ``True``, performs gradient ascent.
    capturable : bool, default=False
        Enables graph-capturable state handling where supported.
    differentiable : bool, default=False
        If ``True``, records optimizer operations in the autograd graph.
    """
    _default = toptim.NAdam


class RAdam(BioOptimizer):
    """
    Rectified Adam optimizer.

    RAdam modifies Adam by accounting for the variance of the adaptive
    learning rate during the early stages of optimization. Adam's
    adaptive second-moment estimate can be poorly behaved when only a
    small number of samples have contributed to it; RAdam dynamically
    rectifies this effect based on the estimated length of the
    available variance.

    Parameters
    ----------
    lr : float, default=1e-3
        Learning rate.
    betas : tuple of float, default=(0.9, 0.999)
        Decay rates for the first- and second-moment estimates.
    eps : float, default=1e-8
        Numerical-stability constant.
    weight_decay : float, default=0
        Weight-decay coefficient.
    decoupled_weight_decay : bool, default=False
        If ``True``, applies weight decay independently of the adaptive
        gradient update.
    foreach : bool or None, default=None
        Whether to use the multi-tensor implementation when available.
    maximize : bool, default=False
        If ``True``, performs gradient ascent.
    capturable : bool, default=False
        Enables graph-capturable state handling where supported.
    differentiable : bool, default=False
        If ``True``, records optimizer operations in the autograd graph.
    """
    _default = toptim.RAdam


class RMSprop(BioOptimizer):
    """
    Root Mean Square Propagation optimizer.

    RMSprop maintains an exponentially weighted average of squared
    gradients and divides the gradient by the square root of this
    running average:

    ``v_t = alpha * v_{t-1} + (1 - alpha) * g_t²``

    ``θ_t = θ_{t-1} - lr * g_t / (sqrt(v_t) + eps)``.

    An optional momentum term can further smooth the parameter updates.

    Parameters
    ----------
    lr : float, default=1e-2
        Learning rate.
    alpha : float, default=0.99
        Smoothing coefficient for the running squared-gradient average.
    eps : float, default=1e-8
        Numerical-stability constant.
    weight_decay : float, default=0
        L2 weight-decay coefficient.
    momentum : float, default=0
        Momentum coefficient. ``0`` disables momentum.
    centered : bool, default=False
        If ``True``, normalizes using an estimate of gradient variance
        rather than only the second moment.
    foreach : bool or None, default=None
        Whether to use the multi-tensor implementation when available.
    maximize : bool, default=False
        If ``True``, performs gradient ascent.
    differentiable : bool, default=False
        If ``True``, records optimizer operations in the autograd graph.
    """
    _default = toptim.RMSprop


class Rprop(BioOptimizer):
    """
    Resilient Backpropagation optimizer.

    Rprop adapts the magnitude of each parameter update according to
    changes in the sign of its gradient rather than its absolute
    magnitude. If the gradient retains the same sign between steps,
    the update magnitude increases; if the sign changes, the magnitude
    decreases.

    This makes Rprop substantially less sensitive to the scale of the
    gradients than magnitude-based optimizers.

    Parameters
    ----------
    lr : float, default=1e-2
        Initial update magnitude for each parameter.
    etas : tuple of float, default=(0.5, 1.2)
        Multiplicative factors used to decrease or increase the update
        magnitude after a gradient-sign change or continuation.
    step_sizes : tuple of float, default=(1e-6, 50)
        Minimum and maximum allowed update magnitudes.
    foreach : bool or None, default=None
        Whether to use the multi-tensor implementation when available.
    maximize : bool, default=False
        If ``True``, performs gradient ascent.
    """
    _default = toptim.Rprop


class SGD(BioOptimizer):
    """
    Stochastic Gradient Descent optimizer.

    SGD updates parameters directly using their gradients:

    ``θ_t = θ_{t-1} - lr * g_t``.

    Optional momentum maintains a velocity that combines the current
    gradient with previous updates:

    ``v_t = momentum * v_{t-1} + g_t``.

    Nesterov momentum modifies this update using a look-ahead gradient.
    Weight decay can additionally regularize the parameters.

    Parameters
    ----------
    lr : float
        Learning rate controlling the magnitude of each update.
    momentum : float, default=0
        Momentum factor. ``0`` disables momentum.
    dampening : float, default=0
        Dampening applied to the contribution of the current gradient
        when momentum is enabled.
    weight_decay : float, default=0
        L2 weight-decay coefficient.
    nesterov : bool, default=False
        If ``True``, uses Nesterov momentum. Requires ``momentum > 0``
        and ``dampening == 0``.
    maximize : bool, default=False
        If ``True``, performs gradient ascent.
    foreach : bool or None, default=None
        Whether to use the multi-tensor implementation when available.
    differentiable : bool, default=False
        If ``True``, records optimizer operations in the autograd graph.
    fused : bool or None, default=None
        Whether to use the fused implementation when supported.
    """
    _default = toptim.SGD


class SparseAdam(BioOptimizer):
    """
    Adam optimizer for parameters with sparse gradients.

    SparseAdam applies Adam-style first- and second-moment updates only
    to entries for which the gradient is present. This avoids performing
    dense optimizer-state updates for parameters whose gradients are
    sparse.

    It is intended for parameters such as suitable embedding tables
    that produce sparse gradients. It should not be used with ordinary
    dense gradients.

    Parameters
    ----------
    lr : float, default=1e-3
        Learning rate.
    betas : tuple of float, default=(0.9, 0.999)
        Decay rates for the first- and second-moment estimates.
    eps : float, default=1e-8
        Numerical-stability constant added to the denominator.
    maximize : bool, default=False
        If ``True``, performs gradient ascent.
    """
    _default = toptim.SparseAdam

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()